In [5]:
import numpy as np
import pandas as pd
print(np.__version__)
print(pd.__version__)
import jax
print(jax.__version__)
print(jax.devices())

1.26.4
2.3.3
0.6.2
[CudaDevice(id=0)]


Importing cellular automata & optimization classes, and other stuff

In [ ]:
import os
import sys
import shutil

from typing import List, Type, Callable, Dict
from numpy import int32
from numpy._typing import NDArray
import importlib

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))

#from algorithm.blender import Lattice, clear_initial
from algorithm.genetic import Optimizer, Mutator, RulesetMutator, ArbitraryRulesetMutator, MutationSet
from algorithm.objectives import surface_to_vol

import numpy as np
import pandas as pd

import time

Setting up optimizer and data logging code

In [7]:
def log_mutation(data_list: List[Dict], mutations: List[MutationSet], objective_val: float):
    """
    Given the data list reference, the mutation set, and the objective value after applying it, add it to the data logging list
    """
    ic_cell_pos = []
    ic_state_old = []
    ic_state_new = []
    srt_cell_pos = []
    srt_state_old = []
    srt_state_new = []

    """
    Important difference from original genetic algorithm: 
    ic_mut is a 4 membered list showing the mutated position in the IC.
    Example: [0,0,0,1], meaning that at IC pos (0,0) the state 0 is modified to be 1.
    srt_mut is a 6 membered list showing the mutated position in the ruleset.
    Example: [0,0,0,0,0,1], meaning that at SRT rule pos (0,0,0) index 0 the rule 0 is modified to be 1.
    For a 2 state SRT there are 18 different cells for mutation: 0 - 17.
    """
    for ic_mut in mutations.ic_mutations:
        ic_cell_pos.append(tuple(ic_mut[0:2]))
        ic_state_old.append(ic_mut[2])
        ic_state_new.append(ic_mut[3])
    for srt_mut in mutations.srt_mutations:
        srt_cell_pos.append(tuple(srt_mut[0:4]))
        srt_state_old.append(srt_mut[4])
        srt_state_new.append(srt_mut[5])
    
    data_list.append({
        "ic_cell_pos": np.array(ic_cell_pos), 
        "ic_state_old": np.array(ic_state_old), 
        "ic_state_new": np.array(ic_state_new), 
        "srt_cell_pos": np.array(srt_cell_pos), 
        "srt_state_old": np.array(srt_state_old), 
        "srt_state_new": np.array(srt_state_new), 
        "objective": objective_val,
    })

def run_experiment(iters: int, grid_sz: int, ruleset_mutator_class: Type[Mutator], rule_set: List[NDArray], opt_func: Callable[[NDArray[int32]], int],
                   ic_num_mutate: int, srt_num_mutate: int, rule_mutate_prob: float, strict: bool = False, num_strict: bool = True, ic_enable: bool = True, srt_enable: bool = True):
    """
    Runs an experiment with the below hyperparameters:

    :param iters: The number of iterations the mutation algorithm (updating both IC and SRT) is going to run for
    :param grid_sz: The size of the square grid that we're going to update each iteration
    :param ruleset_mutator_class: The type of mutator used: RulesetMutator or ArbitraryRulesetMutator
    :param rule_set: The list of possible rulesets if RulesetMutator is used
    :param opt_func: The functions that gives the performance metric we're going to optimize
    :param srt_num_mutate: The number of SRT cells for which we're going to mutate the rule applied, each iteration
    :param ic_num_mutate: The number of IC cells for which we're going to mutate the rule applied, each iteration
    :param rule_mutate_prob: The probability, for each neighbor state tensor of the rule of a cell that's selected to be mutated, the final state is mutated
    :param strict: Whether SRT mutations are chosen by cell then rule or by rule directly; for more info, see mutation.py
    :param num_strict: Whether the number of SRT/IC cells mutated will be constant per iteration or variable; for more info, see mutation.py; incompatible with RulesetMutator
    :param ic_enable: Whether the IC will be mutated
    :param srt_enable: Whether the SRT will be mutated
    """
    # RESOLVED: separate SRT and IC mutations to have a certain number of each
    # RESOLVED: add a flag to enable doing only SRT or only IC mutations in an iteration (in optimizer step, and then propagate into mutator)
    ruleset_mutator = ruleset_mutator_class(rules=rule_set, grid_size=grid_sz, mutate_p=1/(grid_sz**2) * (srt_num_mutate+ic_num_mutate), rule_mutate_p=rule_mutate_prob, strict=strict, num_strict = num_strict, ic_ct = ic_num_mutate, srt_ct=srt_num_mutate, ic_enable=ic_enable, srt_enable=srt_enable)

    optim = Optimizer(mutator=ruleset_mutator, objective=lambda grid: opt_func(grid))

    """
    Pandas Dataframe used to log experiment data is:

    ic_cell_pos (np.array) | ic_state_old (np.array) | ic_state_new (np.array) | srt_cell_pos (np.array) | srt_state_old (np.array) | srt_state_new (np.array) | objective (float)
    
    etc.

    initial state for IC is in entry 0 in ic_state_old, and SRT is in entry 0 in srt_state_old

    ic and srt mutation cell positions and states can have an extra dimension in the beginning to indicate they are batch updates
    """

    init_state = optim.state
    
    data_list = [{"ic_cell_pos": grid_sz, 
                  "ic_state_old": init_state.initial, 
                  "ic_state_new": None, 
                  "srt_cell_pos": -1, 
                  "srt_state_old": init_state.rules, 
                  "srt_state_new": None, 
                  "objective": 0}]

    for it in range(iters):
        # if (it%20 == 0):
        #     print(f"Iteration {it}")
        # print(f"On iteration {it+1}...")
        accepted, new, old, mutations = optim.step()
        # data logging
        log_mutation(data_list, mutations, optim.objvalue)
        # if accepted:
        #     print("Got a better state!", optim.objvalue)

    # print(data_list)
    df = pd.DataFrame(data_list)
    # print(df)
    return df

timelogs = []

def repeat_experiment(experiment_name: str, num_expers: int, *args):
    """
    Perform (sequentially) multiple experiments that return a Pandas DataFrame and save all the data

    :param experiment_name: The name of the experiment to save the file
    :param num_expers: Number of times to run the experiment (and save all the data in one file)
    :param *args: The arguments to be passed to the experiment function
    """
    for i in range(num_expers):
        init = time.time()
        print(f'REPETITION {i}')
        ret_data = run_experiment(*args)
        timelogs.append(time.time() - init)
        ret_data.to_hdf(f'gpu_test/{experiment_name}_{i}.h5', key='data', mode='a')


Setting up experiments and gathering data

In [8]:
ITERATIONS_SET = [50, 100, 200, 500]
GRID_SIZE_SET = [10, 20, 32, 64, 100]
NUM_REPEAT = 20
EXPERIMENT_NAME = "default"

for iters in ITERATIONS_SET:
    for grid_sz in GRID_SIZE_SET:
        print(f"RUNNING EXPERIMENT {EXPERIMENT_NAME} WITH {iters} ITERATIONS AND {grid_sz} SIZE GRID")
        #Total param is set default to false meaning that the probability is not the total probability; 
        #so that the program aligns closer to that of the original genetic algorithm
        
        #def probability for Strict Mode = 2/3; def probability for Non-Strict Mode = 5/384
        repeat_experiment(f"{EXPERIMENT_NAME}_{iters}ITERS_{grid_sz}GRID", NUM_REPEAT, iters, grid_sz, ArbitraryRulesetMutator, [[0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0],[0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]], surface_to_vol, grid_sz, grid_sz**2, 2/3, True, True, True, True)
        print(f"Finished reps for [{iters} ITERS, {grid_sz} GSIZE] in {timelogs[-(NUM_REPEAT):]}")

RUNNING EXPERIMENT default WITH 50 ITERATIONS AND 10 SIZE GRID
REPETITION 0


/tmp/ipykernel_227439/4138432953.py:115: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{experiment_name}_{i}.h5', key='data', mode='a')


REPETITION 1


/tmp/ipykernel_227439/4138432953.py:115: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{experiment_name}_{i}.h5', key='data', mode='a')


REPETITION 2


...

REPETITION 18


/tmp/ipykernel_227439/4138432953.py:115: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{experiment_name}_{i}.h5', key='data', mode='a')


REPETITION 19


/tmp/ipykernel_227439/4138432953.py:115: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{experiment_name}_{i}.h5', key='data', mode='a')


Finished reps for [500 ITERS, 100 GSIZE] in [193.4978849887848, 193.0509524345398, 191.77187705039978, 191.919584274292, 192.72678565979004, 191.31312656402588, 191.1543562412262, 191.34965467453003, 190.98558020591736, 191.47919750213623, 192.56949973106384, 190.56789445877075, 192.17672204971313, 191.83705806732178, 192.3541021347046, 193.240065574646, 193.25396013259888, 192.18212580680847, 191.98502206802368, 192.95477652549744]


In [9]:
#GPU MutVar Trial Timelogs
print(timelogs)

[2.06770920753479, 2.0903923511505127, 1.09812331199646, 1.9817068576812744, 1.8267021179199219, 1.778886318206787, 2.0616867542266846, 1.8236174583435059, 1.8095519542694092, 1.8657310009002686, 2.0206539630889893, 1.8524329662322998, 1.8241753578186035, 2.021075963973999, 1.952627182006836, 1.8278017044067383, 1.948638677597046, 1.9983680248260498, 2.1184682846069336, 0.894218921661377, 2.4237537384033203, 2.1201422214508057, 2.247339963912964, 2.0341720581054688, 2.1593000888824463, 2.0607781410217285, 2.1491410732269287, 1.9591984748840332, 2.1159508228302, 2.106137990951538, 2.0737390518188477, 1.9602839946746826, 2.0103890895843506, 1.9531402587890625, 0.9647059440612793, 2.002030849456787, 1.9774065017700195, 1.932631015777588, 1.911156415939331, 1.9978854656219482, 3.528498888015747, 2.3322999477386475, 2.262760877609253, 2.318164348602295, 2.3389549255371094, 2.6119539737701416, 2.3485326766967773, 2.284499168395996, 1.3476459980010986, 2.3385396003723145, 2.30951189994812, 2.

In [10]:
ITERATIONS_SET = [50, 100, 200, 500]
GRID_SIZE_SET = [10, 20, 32, 64]
NUM_REPEAT = 20
EXPERIMENT_NAME = "comparative" #Directly comparable to naive implementation

for iters in ITERATIONS_SET:
    for grid_sz in GRID_SIZE_SET:
        print(f"RUNNING EXPERIMENT {EXPERIMENT_NAME} WITH {iters} ITERATIONS AND {grid_sz} SIZE GRID")
        #Total param is set default to false meaning that the probability is not the total probability; 
        #so that the program aligns closer to that of the original genetic algorithm
        
        #def probability for Strict Mode = 2/3; def probability for Non-Strict Mode = 5/384
        repeat_experiment(f"{EXPERIMENT_NAME}_{iters}ITERS_{grid_sz}GRID", NUM_REPEAT, iters, grid_sz, ArbitraryRulesetMutator, [[0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0],[0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]], surface_to_vol, 10, 10, 2/3, True, False, True, True)
        print(f"Finished reps for [{iters} ITERS, {grid_sz} GSIZE] in {timelogs[-(NUM_REPEAT):]}")

RUNNING EXPERIMENT comparative WITH 50 ITERATIONS AND 10 SIZE GRID
REPETITION 0


/tmp/ipykernel_227439/4138432953.py:115: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{experiment_name}_{i}.h5', key='data', mode='a')


REPETITION 1


/tmp/ipykernel_227439/4138432953.py:115: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{experiment_name}_{i}.h5', key='data', mode='a')


REPETITION 2


...

REPETITION 18


/tmp/ipykernel_227439/4138432953.py:115: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{experiment_name}_{i}.h5', key='data', mode='a')


REPETITION 19


/tmp/ipykernel_227439/4138432953.py:115: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{experiment_name}_{i}.h5', key='data', mode='a')


Finished reps for [500 ITERS, 64 GSIZE] in [85.07488298416138, 90.64515709877014, 90.97000098228455, 86.243816614151, 92.56007170677185, 91.80905079841614, 93.89029288291931, 94.23044037818909, 95.26308035850525, 95.20237946510315, 94.48172044754028, 94.7291784286499, 93.94475197792053, 94.90294861793518, 92.68941259384155, 94.50351524353027, 94.1601619720459, 94.11369299888611, 94.80414962768555, 94.87396478652954]


In [11]:
#GPU Naive-Comparable Trial Timelogs
print(timelogs)

[2.06770920753479, 2.0903923511505127, 1.09812331199646, 1.9817068576812744, 1.8267021179199219, 1.778886318206787, 2.0616867542266846, 1.8236174583435059, 1.8095519542694092, 1.8657310009002686, 2.0206539630889893, 1.8524329662322998, 1.8241753578186035, 2.021075963973999, 1.952627182006836, 1.8278017044067383, 1.948638677597046, 1.9983680248260498, 2.1184682846069336, 0.894218921661377, 2.4237537384033203, 2.1201422214508057, 2.247339963912964, 2.0341720581054688, 2.1593000888824463, 2.0607781410217285, 2.1491410732269287, 1.9591984748840332, 2.1159508228302, 2.106137990951538, 2.0737390518188477, 1.9602839946746826, 2.0103890895843506, 1.9531402587890625, 0.9647059440612793, 2.002030849456787, 1.9774065017700195, 1.932631015777588, 1.911156415939331, 1.9978854656219482, 3.528498888015747, 2.3322999477386475, 2.262760877609253, 2.318164348602295, 2.3389549255371094, 2.6119539737701416, 2.3485326766967773, 2.284499168395996, 1.3476459980010986, 2.3385396003723145, 2.30951189994812, 2.